In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cf
import matplotlib as mpl
from pylab import *

ERROR 1: PROJ: proj_create_from_database: Open of /nesi/project/niwa00018/queenle/ml_env_v2/share/proj failed


In [2]:
'''
Create vegetation land masks
'''
ERA5 = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/ERA5/model-style/6month_fire_season/FWI_6month_fire_season_ERA5_resolution.nc')
d4pdf = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/d4pdf/FWI/6month_fire_season/BC_wind_FWI/BC_wind_FWI_HPB_ensMean.nc')
cam5 = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/CAM5-1-1/6month_fire_season/FWI_All_ensMean.nc')

vege_ds = xr.open_dataset('/nesi/project/niwa00015/queenle/data/ERA5/monthly/vegetation/1951-2021.nc')
vege_ds['longitude'] = [lon + 360 for lon in vege_ds.longitude.data]
era5_combined = (vege_ds.lai_hv + vege_ds.lai_lv).max('time')
era5_vege_mask = era5_combined.where(era5_combined>3.5)

vege_ds_coarsened = vege_ds.coarsen(latitude=3,longitude=3,boundary='pad').mean()

cam5_vege = vege_ds_coarsened.interp(longitude=cam5.longitude, latitude=cam5.latitude, method="linear")
cam5_combined = (cam5_vege.lai_hv + cam5_vege.lai_lv).max('time')
cam5_vege_mask = cam5_combined.where(cam5_combined>3.5)

d4pdf_vege = vege_ds_coarsened.interp(longitude=d4pdf.longitude, latitude=d4pdf.latitude, method="linear")
d4pdf_combined = (d4pdf_vege.lai_hv + d4pdf_vege.lai_lv).max('time')
d4pdf_vege_mask = d4pdf_combined.where(d4pdf_combined>3.5)

box = [(32,55),(232,256)]

def get_linregress_slope(arr):
    if (np.isnan(arr)).any():
        result = np.nan
    else:
        result = np.polyfit([i for i in range(len(arr))],arr,deg=1)[0]
        
    return(result)


In [4]:
def process_data(da,era5_da,start,end,vege_mask):
    
    da = da.sel(latitude=slice(box[0][0],box[0][1]),longitude=slice(box[1][0],box[1][1]))
    da = da.sel(time=slice(start,end))
    da = da.where(vege_mask>0)
    da = da.where(~era5_da.isnull())
    
    da_trend = xr.apply_ufunc(get_linregress_slope, da, input_core_dims=[["time"]], vectorize = True)
    
    return(da_trend)

In [5]:
'''
Both models, entire region
'''

RESULTS_DICT = {'model':[],'time_window':[],'ant_low':[],'ant_contrib (% FWI/yr)':[],'ant_up':[],'nat_low':[],'nat_contrib (% FWI/yr)':[],'nat_up':[]}


for model,start,end in [['d4pdf','1951','2021'],['d4pdf','1959','2014'],['d4pdf','1984','2014'],['cam5','1959','2014'],['cam5','1984','2014']]:
    
    if model == 'd4pdf':
        
        trunc = 60
        vege_mask = d4pdf_vege_mask
        
        Ant_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/d4pdf/FWI/6month_fire_season/BC_wind_FWI/BC_wind_FWI_Ant_ensMean.nc')
        Nat_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/d4pdf/FWI/6month_fire_season/BC_wind_FWI/BC_wind_FWI_HPB_NAT_ensMean.nc')


    elif model == 'cam5':
        
        trunc = 30
        vege_mask = cam5_vege_mask
        
        Ant_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/CAM5-1-1/6month_fire_season/FWI_Ant_ensMean.nc')
        Nat_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/CAM5-1-1/6month_fire_season/FWI_Nat_ensMean.nc')

        
    # load scaling factors
    SFs = pd.read_csv('/home/queenle/fire_attribution/result_files/'+model+'_scan_beta_results_'+start+'_'+end+'.csv')
    ANT_SF_low = SFs[SFs['Truncation']==trunc]['ant_lower'].values[0]
    ANT_SF = SFs[SFs['Truncation']==trunc]['ant_best'].values[0]
    ANT_SF_up = SFs[SFs['Truncation']==trunc]['ant_upper'].values[0]
    
    NAT_SF_low = SFs[SFs['Truncation']==trunc]['nat_lower'].values[0]
    NAT_SF = SFs[SFs['Truncation']==trunc]['nat_best'].values[0]
    NAT_SF_up = SFs[SFs['Truncation']==trunc]['nat_upper'].values[0]

    # load observations
    era5_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/ERA5/model-style/6month_fire_season/'+model+'_resolution/FWI_6month_fire_season_'+model+'_resolution.nc')
    era5_da = era5_da.sel(latitude=slice(box[0][0],box[0][1]),longitude=slice(box[1][0],box[1][1]))
    era5_da = era5_da.sel(time=slice(start,end))
    era5_trend = xr.apply_ufunc(get_linregress_slope, era5_da, input_core_dims=[["time"]], vectorize = True)
    
    weights = np.cos(np.deg2rad(era5_trend.latitude))
    weights.name = "weights"
    weighted_era5_trend = era5_trend.weighted(weights)


    # load response patterns
    Ant_trend = process_data(Ant_da,era5_da,start,end,vege_mask)
    weights = np.cos(np.deg2rad(Ant_trend.latitude))
    weights.name = "weights"
    weighted_Ant_trend = Ant_trend.weighted(weights)

    Nat_trend = process_data(Nat_da,era5_da,start,end,vege_mask)
    weights = np.cos(np.deg2rad(Nat_trend.latitude))
    weights.name = "weights"
    weighted_Nat_trend = Nat_trend.weighted(weights)

    
    # calc contributions
    
    obs_trend = weighted_era5_trend.mean(['latitude','longitude']).values[0]
    ant_trend = weighted_Ant_trend.mean(['latitude','longitude']).values[0]
    nat_trend = weighted_Nat_trend.mean(['latitude','longitude']).values[0]

    ant_contrib_low = (ant_trend * ANT_SF_low / obs_trend) * 100 # percent of regional mean FWI per year
    ant_contrib = (ant_trend * ANT_SF / obs_trend) * 100 # percent of regional mean FWI per year
    ant_contrib_up = (ant_trend * ANT_SF_up / obs_trend) * 100 # percent of regional mean FWI per year

    nat_contrib_low = (nat_trend * NAT_SF_low / obs_trend) * 100 # percent of regional mean FWI per year
    nat_contrib = (nat_trend * NAT_SF / obs_trend) * 100
    nat_contrib_up = (nat_trend * NAT_SF_up / obs_trend) * 100 # percent of regional mean FWI per year


    RESULTS_DICT['model'].append(model)
    RESULTS_DICT['time_window'].append(f'{start}-{end}')

    RESULTS_DICT['ant_low'].append(ant_contrib_low)
    RESULTS_DICT['ant_contrib (% FWI/yr)'].append(ant_contrib)
    RESULTS_DICT['ant_up'].append(ant_contrib_up)
    
    RESULTS_DICT['nat_low'].append(nat_contrib_low)
    RESULTS_DICT['nat_contrib (% FWI/yr)'].append(nat_contrib)
    RESULTS_DICT['nat_up'].append(nat_contrib_up)
    

df = pd.DataFrame.from_dict(RESULTS_DICT)

In [6]:
df

,model,time_window,ant_low,ant_contrib (% FWI/yr),ant_up,nat_low,nat_contrib (% FWI/yr),nat_up
0,d4pdf,1951-2021,63.192040,102.997729,142.803418,-28.643597,-33.261869,-37.880141
1,d4pdf,1959-2014,47.397264,80.765567,114.133869,-10.964466,-12.949033,-14.933600
2,d4pdf,1984-2014,-1.653207,15.723739,33.100685,22.766867,26.168042,29.569216
3,cam5,1959-2014,159.659657,188.444815,217.229961,-25.926318,-30.861041,-35.795764
4,cam5,1984-2014,65.654133,88.233161,110.812196,9.235612,11.293688,13.351765


In [15]:
df

,model,time_window,ant_low,ant_contrib (% FWI/yr),ant_up,nat_low,nat_contrib (% FWI/yr),nat_up
0,d4pdf,1951-2021,60.301414,98.286252,136.271089,-27.344532,-31.753353,-36.162174
1,d4pdf,1959-2014,46.487122,79.214671,111.942221,-11.179448,-13.202927,-15.226406
2,d4pdf,1984-2014,-1.696198,16.132631,33.961461,21.846528,25.110211,28.373895
3,cam5,1959-2014,171.324379,202.212578,233.100765,-27.460639,-32.687400,-37.914160
4,cam5,1984-2014,72.825000,97.870153,122.915312,5.941512,7.265526,8.589541


In [4]:
subregions = {'Canada':[49,55,232,255],'NW USA':[42,49,232,255],
              'California':[32,42,232,244],'Four Corners':[32,41,246,256],'entire_region':[]}


In [5]:
RESULTS_DICT = {'model':[],'subregion':[],'ant_contrib (% FWI/yr)':[],'nat_contrib (% FWI/yr)':[]}

In [6]:
'''
d4PDF-G
'''

model = 'd4pdf'
start = '1959'
end = '2014' 
trunc = 60

# load scaling factors
SFs = pd.read_csv('/home/queenle/fire_attribution/result_files/'+model+'_scan_beta_results_'+start+'_'+end+'.csv')
ANT_SF = SFs[SFs['Truncation']==trunc]['ant_best'].values[0]
NAT_SF = SFs[SFs['Truncation']==trunc]['nat_best'].values[0]

# load observations
era5_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/ERA5/model-style/6month_fire_season/d4pdf_resolution/FWI_6month_fire_season_d4pdf_resolution.nc')
era5_da = era5_da.sel(latitude=slice(box[0][0],box[0][1]),longitude=slice(box[1][0],box[1][1]))
era5_trend = xr.apply_ufunc(get_linregress_slope, era5_da, input_core_dims=[["time"]], vectorize = True)

# load response patterns
Ant_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/d4pdf/FWI/6month_fire_season/BC_wind_FWI/BC_wind_FWI_Ant_ensMean.nc')
Ant_da = Ant_da.sel(latitude=slice(box[0][0],box[0][1]),longitude=slice(box[1][0],box[1][1]))
Ant_da = Ant_da.sel(time=slice(start,end))
Ant_da = Ant_da.where(d4pdf_vege_mask>0)
Ant_da = Ant_da.where(~era5_da.isnull())
Ant_trend = xr.apply_ufunc(get_linregress_slope, Ant_da, input_core_dims=[["time"]], vectorize = True)

Nat_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/d4pdf/FWI/6month_fire_season/BC_wind_FWI/BC_wind_FWI_HPB_NAT_ensMean.nc')
Nat_da = Nat_da.sel(latitude=slice(box[0][0],box[0][1]),longitude=slice(box[1][0],box[1][1]))
Nat_da = Nat_da.sel(time=slice(start,end))
Nat_da = Nat_da.where(d4pdf_vege_mask>0)
Nat_da = Nat_da.where(~era5_da.isnull())
Nat_trend = xr.apply_ufunc(get_linregress_slope, Nat_da, input_core_dims=[["time"]], vectorize = True)

for i,subregion in enumerate(subregions):
    print(subregion)

    if subregion != 'entire_region':
        sr = subregions[subregion]
        obs_sr_mean = era5_da.sel(latitude=slice(sr[0],sr[1]),longitude=slice(sr[2],sr[3])).mean(['latitude','longitude','time']).values[0]
        obs_sr_trend = era5_trend.sel(latitude=slice(sr[0],sr[1]),longitude=slice(sr[2],sr[3])).mean(['latitude','longitude']).values[0]
        ant_sr_trend = Ant_trend.sel(latitude=slice(sr[0],sr[1]),longitude=slice(sr[2],sr[3])).mean(['latitude','longitude']).values[0]
        nat_sr_trend = Nat_trend.sel(latitude=slice(sr[0],sr[1]),longitude=slice(sr[2],sr[3])).mean(['latitude','longitude']).values[0]

    else:
        obs_sr_mean = era5_da.mean(['latitude','longitude','time']).values[0]
        obs_sr_trend = era5_trend.mean(['latitude','longitude']).values[0]
        ant_sr_trend = Ant_trend.mean(['latitude','longitude']).values[0]
        nat_sr_trend = Nat_trend.mean(['latitude','longitude']).values[0]

    ant_contrib = (ant_sr_trend * ANT_SF / obs_sr_trend) * 100 # percent of regional mean FWI per year
    nat_contrib = (nat_sr_trend * NAT_SF / obs_sr_trend) * 100
    
    RESULTS_DICT['model'].append(model)
    RESULTS_DICT['subregion'].append(subregion)
    RESULTS_DICT['ant_contrib (% FWI/yr)'].append(ant_contrib)
    RESULTS_DICT['nat_contrib (% FWI/yr)'].append(nat_contrib)
    
    print(f'ERA5 mean: {obs_sr_mean}')
    print(f'ant_contrib: {ant_contrib}')
    print(f'nat_contrib: {nat_contrib}')
    
    print(f'ANT trend: {ant_sr_trend}')
    print(f'NAT trend: {nat_sr_trend}')
    print(f'ANT_trend*ANT_scaling: {ant_sr_trend * ANT_SF}')
    print(f'NAT_trend*NAT_scaling: {nat_sr_trend * NAT_SF}')
    print(f'{obs_sr_trend} = {ant_sr_trend * ANT_SF} + {nat_sr_trend * NAT_SF} = {ant_sr_trend * ANT_SF + nat_sr_trend * NAT_SF}')
    print('\n')



Canada
ERA5 mean: 11.352759985434812
ant_contrib: -25.955704022231263
nat_contrib: -39.96874807020505
ANT trend: -0.0006782447753798064
NAT trend: -0.0028836396121534025
ANT_trend*ANT_scaling: -0.0011377419529977207
NAT_trend*NAT_scaling: -0.0017519895221998231
0.004383398547090982 = -0.0011377419529977207 + -0.0017519895221998231 = -0.002889731475197544


NW USA
ERA5 mean: 25.057415850865738
ant_contrib: 101.34921155310433
nat_contrib: -19.320517857577418
ANT trend: 0.02425554756816057
NAT trend: -0.012766629807179389
ANT_trend*ANT_scaling: 0.040688192615674024
NAT_trend*NAT_scaling: -0.007756517687478753
0.040146530981501005 = 0.040688192615674024 + -0.007756517687478753 = 0.03293167492819527


California
ERA5 mean: 38.91772534112453
ant_contrib: 123.99968412820512
nat_contrib: -24.64809217055561
ANT trend: 0.051431789753082796
NAT trend: -0.0282267685111276
ANT_trend*ANT_scaling: 0.08627579163742542
NAT_trend*NAT_scaling: -0.017149508721072692
0.06957742855878858 = 0.086275791637425

In [7]:
'''
CAM5
'''

model = 'cam5'
start = '1959'
end = '2014'
trunc = 30

print(model)
print('\n')

# load scaling factors
SFs = pd.read_csv('/home/queenle/fire_attribution/result_files/'+model+'_scan_beta_results_'+start+'_'+end+'.csv')
ANT_SF = SFs[SFs['Truncation']==trunc]['ant_best'].values[0]
NAT_SF = SFs[SFs['Truncation']==trunc]['nat_best'].values[0]

print(f'ant scaling factor: {ANT_SF}')
print(f'nat scaling factor: {NAT_SF}\n')

# load observations
era5_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/ERA5/model-style/6month_fire_season/cam5_resolution/FWI_6month_fire_season_cam5_resolution.nc')
era5_da = era5_da.sel(latitude=slice(box[0][0],box[0][1]),longitude=slice(box[1][0],box[1][1]))
era5_trend = xr.apply_ufunc(get_linregress_slope, era5_da, input_core_dims=[["time"]], vectorize = True)

# load response patterns
Ant_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/CAM5-1-1/6month_fire_season/FWI_Ant_ensMean.nc')
Ant_da = Ant_da.sel(latitude=slice(box[0][0],box[0][1]),longitude=slice(box[1][0],box[1][1]))
Ant_da = Ant_da.sel(time=slice('1959','2014'))
Ant_da = Ant_da.where(cam5_vege_mask>0)
Ant_da = Ant_da.where(~era5_da.isnull())
Ant_trend = xr.apply_ufunc(get_linregress_slope, Ant_da, input_core_dims=[["time"]], vectorize = True)

Nat_da = xr.open_dataarray('/nesi/project/niwa00015/queenle/data/fire/CAM5-1-1/6month_fire_season/FWI_Nat_ensMean.nc')
Nat_da = Nat_da.sel(latitude=slice(box[0][0],box[0][1]),longitude=slice(box[1][0],box[1][1]))
Nat_da = Nat_da.sel(time=slice('1959','2014'))
Nat_da = Nat_da.where(cam5_vege_mask>0)
Nat_da = Nat_da.where(~era5_da.isnull())
Nat_trend = xr.apply_ufunc(get_linregress_slope, Nat_da, input_core_dims=[["time"]], vectorize = True)

for i,subregion in enumerate(subregions):
    print(subregion)

    if subregion != 'entire_region':
        sr = subregions[subregion]
        obs_sr_mean = era5_da.sel(latitude=slice(sr[0],sr[1]),longitude=slice(sr[2],sr[3])).mean(['latitude','longitude','time']).values[0]
        obs_sr_trend = era5_trend.sel(latitude=slice(sr[0],sr[1]),longitude=slice(sr[2],sr[3])).mean(['latitude','longitude']).values[0]
        ant_sr_trend = Ant_trend.sel(latitude=slice(sr[0],sr[1]),longitude=slice(sr[2],sr[3])).mean(['latitude','longitude']).values[0]
        nat_sr_trend = Nat_trend.sel(latitude=slice(sr[0],sr[1]),longitude=slice(sr[2],sr[3])).mean(['latitude','longitude']).values[0]

    else:
        obs_sr_mean = era5_da.mean(['latitude','longitude','time']).values[0]
        obs_sr_trend = era5_trend.mean(['latitude','longitude']).values[0]
        ant_sr_trend = Ant_trend.mean(['latitude','longitude']).values[0]
        nat_sr_trend = Nat_trend.mean(['latitude','longitude']).values[0]

    ant_contrib = (ant_sr_trend * ANT_SF / obs_sr_trend) * 100 # percent of regional mean FWI per year
    nat_contrib = (nat_sr_trend * NAT_SF / obs_sr_trend) * 100
    
    RESULTS_DICT['model'].append(model)
    RESULTS_DICT['subregion'].append(subregion)
    RESULTS_DICT['ant_contrib (% FWI/yr)'].append(ant_contrib)
    RESULTS_DICT['nat_contrib (% FWI/yr)'].append(nat_contrib)
    
    print(f'ERA5 mean: {obs_sr_mean}')
    print(f'ant_contrib: {ant_contrib}')
    print(f'nat_contrib: {nat_contrib}')
    
    print(f'ANT trend: {ant_sr_trend}')
    print(f'NAT trend: {nat_sr_trend}')
    print(f'ANT_trend*ANT_scaling: {ant_sr_trend * ANT_SF}')
    print(f'NAT_trend*NAT_scaling: {nat_sr_trend * NAT_SF}')
    print(f'{obs_sr_trend} = {ant_sr_trend * ANT_SF} + {nat_sr_trend * NAT_SF} = {ant_sr_trend * ANT_SF + nat_sr_trend * NAT_SF}')
    print('\n')


cam5


ant scaling factor: 2.1080191135406494
nat scaling factor: 1.1001224517822266

Canada
ERA5 mean: 11.324571255851563
ant_contrib: -1717.0948745500048
nat_contrib: 134.24358237629963
ANT trend: 0.039472809258781154
NAT trend: -0.005913311993219742
ANT_trend*ANT_scaling: 0.08320943638265499
NAT_trend*NAT_scaling: -0.006505367288134147
-0.004845942854757021 = 0.08320943638265499 + -0.006505367288134147 = 0.07670406909452084


NW USA
ERA5 mean: 23.80739895660812
ant_contrib: 179.077733763259
nat_contrib: -39.8122438779011
ANT trend: 0.06100046705503763
NAT trend: -0.025986138126109486
ANT_trend*ANT_scaling: 0.128590150486926
NAT_trend*NAT_scaling: -0.02858793398764716
0.07180688954715182 = 0.128590150486926 + -0.02858793398764716 = 0.10000221649927885


California
ERA5 mean: 36.7937985280148
ant_contrib: 114.83856547927786
nat_contrib: -32.865832417641634
ANT trend: 0.04930165512583571
NAT trend: -0.027036588640599768
ANT_trend*ANT_scaling: 0.10392883133445101
NAT_trend*NAT_scaling: 

In [16]:
df = pd.DataFrame.from_dict(RESULTS_DICT)

In [17]:
df

,model,subregion,ant_contrib (% FWI/yr),nat_contrib (% FWI/yr)
0,d4pdf,Canada,-25.955704,-39.968748
1,d4pdf,NW USA,101.349212,-19.320518
2,d4pdf,California,123.999684,-24.648092
3,d4pdf,Four Corners,239.977268,-23.205137
4,d4pdf,entire_region,126.428471,-21.072181
5,cam5,Canada,-1717.094875,134.243582
6,cam5,NW USA,179.077734,-39.812244
7,cam5,California,114.838565,-32.865832
8,cam5,Four Corners,98.317622,-5.333653
9,cam5,entire_region,179.264421,-28.977860


In [18]:
df['1959-2014'] = df['ant_contrib (% FWI/yr)']*56
df['1959-2014'] = df['ant_contrib (% FWI/yr)']*56

In [19]:
df

,model,subregion,ant_contrib (% FWI/yr),nat_contrib (% FWI/yr),1959-2014
0,d4pdf,Canada,-25.955704,-39.968748,-1453.519425
1,d4pdf,NW USA,101.349212,-19.320518,5675.555847
2,d4pdf,California,123.999684,-24.648092,6943.982311
3,d4pdf,Four Corners,239.977268,-23.205137,13438.727032
4,d4pdf,entire_region,126.428471,-21.072181,7079.994373
5,cam5,Canada,-1717.094875,134.243582,-96157.312975
6,cam5,NW USA,179.077734,-39.812244,10028.353091
7,cam5,California,114.838565,-32.865832,6430.959667
8,cam5,Four Corners,98.317622,-5.333653,5505.786832
9,cam5,entire_region,179.264421,-28.977860,10038.807578


In [11]:
entire_region = df[df['subregion'].isin(['entire_region'])][['model','ant_contrib (% FWI/yr)','nat_contrib (% FWI/yr)']]

In [12]:
entire_region.columns = ['model','ANT contribution','NAT contribution']

In [38]:
entire_region

,model,ANT contribution,NAT contribution
4,d4pdf,98.286252,-31.753353
9,cam5,179.264421,-28.977860


In [15]:
entire_region

,model,ANT contribution,NAT contribution
4,d4pdf,126.428471,-21.072181
9,cam5,179.264421,-28.977860
